# Compartment - gauss-only pipeline (Kaggle) - train80

Chay **train80** (compound-level 80/20 split) voi package gon `src/`: chi gauss head
(`N(mu, sigma^2)` per role), khong reg/softmax heads, khong MLM warmup.

### Setup / Quick start
1. **Repo**: private (AmnO-O/MoTune) -> set Kaggle Secret `GITHUB_TOKEN` (PAT co quyen doc repo).
   Repo public -> khong can. Muon chay fork rieng: sua `REPO_URL` / `REPO_BRANCH` o cell 1.
2. **Data**: tu dong doc tu Kaggle mount `/kaggle/input/datasets/ieltsmater/compartment/Compartment`.
3. **Weights (tu chon)**: set Kaggle Secret `HF_TOKEN` -> cell push model len HuggingFace
   (mac dinh `AmnO-O/compartment-weights`, co the doi `HF_REPO_ID` o cell 1).
4. **Run**: chay tuan tu TAT CA cac cell tu tren xuong.

### Data mix - bien `TRAIN_MIX` (cell 1)
- `'full'`  **(mac dinh)**: 4 lineages - en-nn + de-nn + en-pv + **de-pv** = **9825 rows / 1221 compound**.
  de-pv (trennbare Verben, mod=verb, head=particle): `_match_german_pv` tim duoc **100% spans**;
  ~965 rows (65%) non-degenerate (particle tach roi) co supervision that su; cac hang fused
  mot-token (`abgehauen`) tu dong bi mask `allowed`, chi lam representation.
- `'default'`: 3 lineages - en-nn + de-nn + en-pv = **8335 rows / 1063 compound**.
- `'nn'`:      chi en-nn + de-nn = **6778 rows / 905 compound** (tat het PV).

Cuoi cell train, log se in `Loaded NNNN total rows across NNNN unique compounds` -
so phai khop voi bang tren.

### Hyperparams
- `CONFIG_FILE` (env var, default ''): de `config/mae_gauss.json` khi muon dung config file
  (freeze 3 / lora 8 / ccc 0.7 / head_lr 2e-4 ...).
- `OVERRIDES` (cell 1): tune nhanh khong can sua file. Thu tu uu tien:
  default < config file < `OVERRIDES` < `TRAIN_EXTRA`.
- `TRAIN_EXTRA` (env var): `--set` bo sung, phan cach bang dau phay.

### Sau khi train (cac cell cuoi)
- Cell 5: in `metrics.json` (val_rho_mod / head / mean tren holdout 20%).
- Cell TRIAL: load `models/best.pt`, score cac file TRIAL that (en-nn-trial / en-pv-trial /
  de-nn-trial / de-pv-trial), ghi `submission/[lang]-[task]-pred.tsv` theo DUNG format submit
  (khong header, moi dong `ContextID <tab> score(s)`), zip thanh `submission.zip`.
  NN: 2 score (mod + head); PV: 1 score (mean cua 2 head).
### Static anchors (combined backend A/B)
- **Mac dinh chay qua OVERRIDES** (cell 1): dien 4 o o group
  `external static anchors`: `model_backend=combined`, `static_span=true`,
  `static_ext_path=/kaggle/working/cc_en_de_300.vec`, `static_ext_dim=300`.
  (Cach khac: `CONFIG_FILE=config/mae_gauss_static.json` - Kaggle Secret hoac Env Var.)
- Cell 2 tu dong build `/kaggle/working/cc_en_de_300.vec` (fastText EN+DE filtered,
  ~3GB download lan dau moi session) khi OVERRIDES/TRAIN_EXTRA chua `static_ext_path`.
  Giu coverage report (rows with a usable static anchor / German %) de mien A/B.
- A/B baseline (static tat, cung nov): `TRAIN_EXTRA='static_span=false,static_ext_path='`
  (hoac de rong 4 o OVERRIDES tren) de so sanh rho.
- Chu y: `scripts/build_static_vec.py` + `config/mae_gauss_static.json` phai duoc
  commit & push len repo truoc - notebook refresh clone o cell 1.
- Doi nguon anchor: doi OVERRIDES `static_ext_path` + `static_ext_dim` tuong ung
  (`cc_en_de_300.vec` -> 300 fastText).
  Cell 2 chon builder theo ten file cua `static_ext_path`.

### Marked arms (input focus A/B)
- **PREFIX PROMPT (chinh, `target_prefix=true`)**: ngay sau `<bos>` prepend
  `<marker> WORD <marker>` - WORD la target surface form cua row (mod/head/
  whole compound) da BPE-tokenize, marker la unused id 7/8/9 (mod/head/pv).
  Backbone thay ca role lan tu` khoi layer 0:
    mod -> `[CLS] <unused0> acid <unused0> The acid rain ...`
    head -> `[CLS] <unused1> rain <unused1> The acid rain ...`
    pv   -> `[CLS] <unused2> acid rain <unused2> The acid rain ...`
- **INLINE (`span_markers=true`)**: cung id pair nhung splice TRONG cau tai
  span's token boundaries (span o trong text, khong lap tu). 2 cai loai tru
  nhau (`Config.reject`). Ca 2 can combined backend + single targets, va cac
  id` duoc chen post-tokenization (mmBERT khong the tokenize `<unusedN>` tu
  string - da kiem tra).
- **Cach chay**: OVERRIDES (cell 1) `model_backend=combined` +
  `target_prefix=true` (hoac `span_markers=true`), hoac
  `CONFIG_FILE=config/mae_gauss_marked.json` (Kaggle Secret / Env Var).
- **READOUT (`prefix_readout`)**: `context` (mac dinh = baseline) pool SPAN
  cua target trong cau (2nd occurrence). `prefix` = pool WORD tokens trong
  prefix prompt (mention, da contextualized sau 22 layers - chay duoc tren
  fused one-token row ma find_spans khong align duoc). `dual` = tron CA 2
  qua learned scalar gate (khoi dau 50/50, train o head_lr). Require
  `target_prefix=true`.
- Row khong align duoc (fused one-token) chi bo` tron/loai; readout giu
  nguyen - masks van pin mod/head span cho pool (dynamic pad).
- Luu y training: backbone/embedding dong bang (embedding_lr=0.0) nen cac id
  unused giu pretrained embedding den khi LoRA phase 2; POSITION-1 marker
  duoc overwrite boi `marker_emb` (learned, head_lr) trong phase 1. Muon ca
  embeddings marker-hoc tu dau: set `embedding_lr` > 0.
- A/B baseline (marked OFF): de rong ca 2 o tren (mac dinh).



In [ ]:
import os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'src' / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- config file (optional) ------------------------------------------------
# ---- config file (optional) ------------------------------------------------
def _env_or_secret(name: str, default: str = '') -> str:
    v = os.environ.get(name, '')
    if v:
        return v
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return default

CONFIG_FILE = _env_or_secret('CONFIG_FILE', '')   # e.g. 'config/mae_gauss_static.json'

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')      # all | train80
if MODE not in ('all', 'train80'):
    MODE = 'train80'
TRAIN_EXTRA = os.environ.get('TRAIN_EXTRA', '')   # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep src.py default) ------------------
# Edit values here to tune a run without touching --set flags below.
OVERRIDES = {
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '18',                # LoRA window: top layers (0 = all)
    'batch_size': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'bin_sigma': '',
    'num_workers': '',
    # --- pv lineage toggles (de '' o day; bat/tat bang TRAIN_MIX duoi) ---
    'en_pv_train': '',                       # non-empty -> FORCE it on regardless of TRAIN_MIX
    'de_pv_train': '',                       # 'de-pv-train.tsv' adds German PV (~65% supervised; fused one-token rows masked)
    # --- dedicated Gaussian exit layers ---
    'gauss_ctx_mod': '',                        # modifier exit (19 = block 18)
    'gauss_ctx_head': '',                       # head exit (20 = block 19)
    'gauss_ctx_pv': '',                         # PV exit (21,22 = blocks 20,21)
    # --- external static anchors (combined backend A/B) ---------------
    # For the static arm set BOTH of these + the vec, e.g.:
    #   'model_backend': 'combined',
    #   'static_span': 'true',
    #   'static_ext_path': '/kaggle/working/cc_en_de_300.vec',
    #   'static_ext_dim': '300',
    'model_backend': '',                # 'combined' + static_span true = static arm
    'static_span': '',
    'static_ext_path': '',              # cell 2 auto-builds this file if needed
    'static_ext_dim': '',

    # --- input focus (marked arms A/B) --------------------------------
    # 'target_prefix': 'true'  = PREFIX PROMPT (primary): right after <bos>
    #   prepend <marker> WORD <marker>, where WORD is the row's own target
    #   surface form BPE-tokenized (mod/head/whole compound) and <marker> is
    #   the unused id 7/8/9 = mod/head/pv. The id + the word are both visible
    #   to the backbone from layer 0:
    #     [CLS] <unused0> acid <unused0> The acid rain fell ...
    # 'span_markers': 'true' = INLINE alternative: same id pair but spliced
    #   INSIDE the sentence at the span's token boundaries (span earlier in
    #   the text, no word duplication). The two are mutually exclusive; both
    #   need combined backend + single targets. Off = today's marker-free input.
    # 'prefix_readout': 'context' = default pooling (2nd occurrence: the target
    #   SPAN inside the sentence). 'prefix' = pool the WORD tokens in the
    #   prefix prompt instead (the mention, fully contextualized after 22
    #   layers; also works on fused one-token rows whose span cum find_spans
    #   fails). 'dual' = blend BOTH via a learned scalar gate (starts 50/50,
    #   trains at head_lr). Requires target_prefix=true.
    'target_prefix': '',
    'span_markers': '',
    'prefix_readout': '',
}


def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out

# ---- data mix: lineages tham gia train80 -----------------------------------
# 'full'    = 4 lineages: en-nn + de-nn + en-pv + de-pv
#            (9825 rows / 1221 c; de-pv trennbare ~965 supervised)
# 'default' = en-nn + de-nn + en-pv                    (8335 rows / 1063 c)
# 'nn'      = en-nn + de-nn only                       (6778 rows / 905 c)
# Cuoi cell train, log in 'Loaded NNNN total rows ...' de xac nhan mix.
TRAIN_MIX = os.environ.get('TRAIN_MIX', 'full')
EXTRA_SETS: list = []
if TRAIN_MIX == 'nn' and not OVERRIDES.get('en_pv_train'):
    EXTRA_SETS += ['en_pv_train=', 'de_pv_train=']   # 'key=' -> empty string = lineage off
if TRAIN_MIX in ('nn', 'default') and not OVERRIDES.get('de_pv_train') and 'de_pv_train=' not in EXTRA_SETS:
    EXTRA_SETS.append('de_pv_train=')                # drop German PV
# 'full' can them gi: de_pv_train='de-pv-train.tsv' la config default cua src/.
# OVERRIDES non-empty ('en_pv_train'/'de_pv_train') thang TRAIN_MIX.

# ---- memory ---------------------------------------------------------------
# expandable segments reduce T4 fragmentation; inherited by the subprocess
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- HuggingFace push (optional) -----------------------------------------
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'AmnO-O/compartment-weights')
HF_PRIVATE = os.environ.get('HF_PRIVATE', 'true').lower() not in ('0','false','no')

def _hf_token() -> str:
    tok = os.environ.get('HF_TOKEN', '')
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return ''

# ---- ensure imports exist (Kaggle already ships them) ---------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy', 'yaml'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)


In [ ]:
# --- Static anchors: build the needed external vec once per session -----------
# Builder is chosen by static_ext_path's FILENAME:
#   cc_en_de_300.vec  -> build_static_vec.py     (fastText EN+DE, 300-d)
# Tat static cho A/B baseline: TRAIN_EXTRA='static_span=false,static_ext_path='
_flag_parts = [p for p in (os.environ.get('TRAIN_EXTRA', '') + ' '
               + ' '.join(f'{k}={v}' for k, v in OVERRIDES.items() if v))
              .replace(',', ' ').split()]
_cfg_static_path = ''
if CONFIG_FILE and Path(CONFIG_FILE).is_file():
    try:
        import json as _json
        _cfg_static_path = _json.load(open(CONFIG_FILE, encoding='utf-8')).get('static_ext_path', '') or ''
    except Exception:
        _cfg_static_path = ''
_pos = [p.split('=', 1)[1] for p in _flag_parts
        if p.startswith('static_ext_path=') and len(p) > len('static_ext_path=')]
STATIC_VEC_PATH = Path(_pos[0]) if _pos else (Path(_cfg_static_path) if _cfg_static_path
                                              else Path('/kaggle/working/cc_en_de_300.vec'))
_static_cfg = bool(CONFIG_FILE and 'static' in CONFIG_FILE)
_static_on = bool(_pos)
_static_off = ('static_span=false' in _flag_parts) or ('static_ext_path=' in _flag_parts)
_builder_map = {
    'cc_en_de_300.vec': REPO / 'scripts' / 'build_static_vec.py',
}

if MODE in ('all', 'train80') and (_static_cfg or _static_on) and not _static_off:
    script = _builder_map.get(STATIC_VEC_PATH.name)
    if not STATIC_VEC_PATH.is_file():
        if script is None:
            raise SystemExit('no builder registered for ' + STATIC_VEC_PATH.name
                             + ' in cell 2 _builder_map.')
        if not script.is_file():
            raise SystemExit(script.name + ' not in repo - commit & push it '
                             '(this notebook refreshes the clone at cell 1).')
        print('\n>>> building', STATIC_VEC_PATH.name, '...')
        subprocess.run([sys.executable, str(script), '--out', str(STATIC_VEC_PATH)],
                       cwd=REPO, check=True)
    size = f'{STATIC_VEC_PATH.stat().st_size / 1e6:.2f} MB' if STATIC_VEC_PATH.is_file() else 'MISSING'
    print('static vec:', STATIC_VEC_PATH, size)
elif MODE in ('all', 'train80'):
    print('static vec: skipped (khong dung static config / da tat static).')


In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# --- train80: single compound-level 80/20 split ---------------------------
# gauss-only package: python -m src.run  (no warmup / no reg or softmax heads)
if MODE in ('all', 'train80'):
    args = [
        sys.executable, '-m', 'src.run', '--device', 'cuda',
        '--set', 'num_workers=0',            # chá»‘ng deadlock Kaggle
        '--set', 'batch_size=64',
        '--set', 'freeze_epochs=3',
        '--set', 'head_lr=2e-4',
    ]

    if CONFIG_FILE:
        args += ['--config', CONFIG_FILE]

    args += list(cfg_sets(*EXTRA_SETS))   # OVERRIDES then TRAIN_MIX over the config file

    if TRAIN_EXTRA:
        args += [x.strip() for x in TRAIN_EXTRA.split(',') if x.strip()]

    mix_rows = {'full': '9825 / 1221 compounds', 'default': '8335 / 1063 compounds',
                'nn': '6778 / 905 compounds'}
    # --- guarantee the external static vec exists BEFORE training ----------
    # Self-healing: even if a prep cell was skipped (e.g. papermill runs only
    # some cells), build whatever file the final args ask for.
    _BUILDERS = {
        'cc_en_de_300.vec': 'build_static_vec.py',
    }
    _need = ''
    for _a in args:
        if _a.startswith('static_ext_path='):
            _v = _a.split('=', 1)[1]
            if _v:
                _need = _v
    if _need:
        _svp = Path(_need)
        if not _svp.is_file():
            _b = _BUILDERS.get(_svp.name)
            if _b is None:
                raise SystemExit('no builder registered for %s - build %s manually first.'
                                 % (_svp.name, _svp))
            _bsp = REPO / 'scripts' / _b
            if not _bsp.is_file():
                raise SystemExit('%s not in repo - commit & push it (notebook refresh clone).' % _b)
            print('\n>>> building', _svp.name, '...')
            subprocess.run([sys.executable, str(_bsp), '--out', str(_svp)],
                           cwd=REPO, check=True)
            print('built:', _svp, '%.2f MB' % (_svp.stat().st_size / 1e6))
        else:
            print('static vec OK:', _svp)
    print('>>> MODE=%s TRAIN_MIX=%s (%s) CONFIG_FILE=%s' % (
        MODE, TRAIN_MIX, mix_rows.get(TRAIN_MIX, '?'), CONFIG_FILE or '(defaults)'))
    run(args)


In [ ]:
# --- Push trained weights to HuggingFace --------------------------------
# Needs only HF_TOKEN (Kaggle Secret or env var) + HF_REPO_ID set above.
# Skipped automatically when HF_REPO_ID is empty.
if HF_REPO_ID and MODE in ('all', 'train80'):
    if not _hf_token():
        print('SKIP push: no HF_TOKEN found (set Kaggle Secret named HF_TOKEN).')
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=_hf_token())
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type='model')
        work = Path('/kaggle/working')
        # upload all model checkpoints
        models_dir = work / 'models'
        if models_dir.is_dir():
            api.upload_folder(
                folder_path=str(models_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='models',
                allow_patterns='*.pt'
            )
            pt_files = list(models_dir.glob('*.pt'))
            print(f'pushed {len(pt_files)} model file(s)')
        # upload key artifacts
        for name in ('config.json', 'metrics.json', 'history.json'):
            src = work / name
            if src.is_file():
                api.upload_file(path_or_fileobj=str(src), path_in_repo=name, repo_id=HF_REPO_ID)
                print(f'pushed {name}')
        print(f'HF repo: https://huggingface.co/{HF_REPO_ID}')
else:
    print('push skipped (HF_REPO_ID empty or MODE=%s)' % MODE)


In [ ]:
import json

work = Path('/kaggle/working')
p = work / 'metrics.json'
if p.is_file():
    print('--- metrics.json (val rho on holdout 20%) ---')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))
else:
    print('no metrics.json yet')


In [ ]:
# --- TRIAL after training: reload best.pt, score the REAL per-lineage trial files ----
# TRAI files la holdout rieng tung lineage (en-nn-trial / en-pv-trial / de-nn-trial /
# de-pv-trial). KHONG re-split train de lam trial - each lineage has its own file.
# Ghi submission theo DUNG format: zip gom cac file [language]-[task]-pred.tsv,
# KHONG header, moi dong: ContextID <tab> score(s) (NN: mod + head; PV: 1 score = mean
# cua 2 head). Moi lineage mot file pred rieng. Tu dong BO QUA neu nochua best.pt.
import sys, json, logging, torch, zipfile
from pathlib import Path

work = Path('/kaggle/working')
ckpt = work / 'models' / 'best.pt'
if not ckpt.is_file():
    for cand in (REPO / 'models' / 'best.pt', REPO / 'output' / 'models' / 'best.pt', Path('output/models/best.pt'), Path('models/best.pt')):
        if cand.is_file():
            ckpt = cand
            work = cand.parent.parent
            break
if not ckpt.is_file():
    print('SKIP trial: chua co models/best.pt (train chua xong).')
else:
    sys.path.insert(0, str(REPO))
    from src.config import Config
    from src.pipeline import _tokenizer
    from src.data import load_trial, CompDataset, collate_comp
    from src.model import apply_lora, build_model
    from src.train import evaluate
    from torch.utils.data import DataLoader
    from scipy.stats import spearmanr

    lg = logging.getLogger('trial')
    cfg_p = work / 'config.json'
    cfg = (Config() if not cfg_p.is_file()
           else Config(**json.loads(cfg_p.read_text(encoding='utf-8'))))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = _tokenizer(cfg, lg)

    # best.pt duoc luu KHI LoRA adapters dang active (trainer._apply_lora wrap
    # top layers TRUOC khi train) -> phai wrap lai nhu the TRUOC load_state_dict,
    # nguoc lai moi lora key se mismatch (RuntimeError missing/unexpected).
    model = build_model(cfg, device)
    apply_lora(model, rank=cfg.lora_rank, alpha=cfg.lora_alpha,
               dropout=cfg.lora_dropout, targets=cfg.lora_targets,
               from_layer=cfg.lora_from_layer)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    lineage = load_trial(cfg)
    sub = work / 'submission'
    sub.mkdir(parents=True, exist_ok=True)
    for key, rows in lineage.items():
        # --- external static anchors: same wiring as trainer ------------------
        static_vec = None
        _sep = (getattr(cfg, 'static_ext_path', None) or '').strip()
        if _sep:
            if not Path(_sep).is_file():
                raise FileNotFoundError('static_ext_path not found: ' + _sep)
            from src.static_vec import StaticVec
            words = set()
            for r in rows:
                for k in ('mod', 'head', 'compound'):
                    w = r.get(k)
                    if w:
                        words.add(w)
            static_vec = StaticVec(_sep, cfg.static_ext_dim, words)
        ds = CompDataset(rows, tok, max_len=cfg.max_context_length,
                         static_vec=static_vec, target_prefix=cfg.target_prefix,
                         span_markers=getattr(cfg, 'span_markers', False))
        loader = DataLoader(ds, batch_size=64, shuffle=False, collate_fn=collate_comp)
        # return_all=True -> (mod, head, mod_y, head_y, mask), row-aligned voi rows.
        mp, hp, pp, my, hy, mask = evaluate(model, loader, device, return_all=True, return_pv=True)
        assert len(mp) == len(rows), (key, len(mp), len(rows))
        pv = bool(rows[0]['is_pv'])
        score = pp if pv else mp   # dedicated overall PV head   # PV: 1 score = mean 2 head
        lines = []
        for r, pm, ph, sc, gm, gh in zip(rows, mp, hp, score, my, hy):
            if pv:
                lines.append('%s\t%.4f' % (r['context_id'], float(sc)))
            else:
                lines.append('%s\t%.4f\t%.4f' % (r['context_id'], float(pm), float(ph)))
        fname = '%s-pred.tsv' % key          # vi du: en-nn-pred.tsv / de-pv-pred.tsv
        (sub / fname).write_text('\n'.join(lines), encoding='utf-8')
        msk = mask.astype(bool)
        if int(msk.sum()) > 0:
            if pv:
                rho = spearmanr(my[msk], score[msk]).correlation
                print('%s: %d rows | rho=%.4f' % (key, len(rows), rho))
            else:
                rho_m = spearmanr(my[msk], mp[msk]).correlation
                rho_h = spearmanr(hy[msk], hp[msk]).correlation
                print('%s: %d rows | rho mod=%.4f head=%.4f avg=%.4f'
                      % (key, len(rows), rho_m, rho_h, (rho_m + rho_h) / 2.0))
        else:
            print('%s: %d rows | khong co label (chi ghi prediction)' % (key, len(rows)))

    zip_p = work / 'submission.zip'          # archive dung format: cac *-pred.tsv o root
    with zipfile.ZipFile(zip_p, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(sub.glob('*-pred.tsv')):
            zf.write(f, arcname=f.name)
    print('submission files ->', sorted(f.name for f in sub.glob('*-pred.tsv')))
    print('zipped ->', zip_p)

In [ ]:
# --- TRIAL METRICS: compare saved predictions with labels when available ---
# This runs after the trial-prediction cell and is safe for unlabeled test files.
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

from src.config import Config
from src.data import load_trial

cfg_p = work / 'config.json'
cfg = Config() if not cfg_p.is_file() else Config(**json.loads(cfg_p.read_text(encoding='utf-8')))
trial_metrics = {}
for key, rows in load_trial(cfg).items():
    pred_path = work / 'submission' / f'{key}-pred.tsv'
    if not pred_path.is_file():
        print(f'{key}: prediction file missing: {pred_path.name}')
        continue

    gold = pd.DataFrame({
        'ContextID': [r['context_id'] for r in rows],
        'mod_avg': [r['mod_avg'] for r in rows],
        'head_avg': [r['head_avg'] for r in rows],
    })
    pred = pd.read_csv(pred_path, sep='\t', header=None, dtype=str)
    if bool(rows[0]['is_pv']):
        pred.columns = ['ContextID', 'pred']
        pred['pred'] = pd.to_numeric(pred['pred'], errors='coerce')
        scored = gold.merge(pred, on='ContextID', how='inner')
        mask = np.isfinite(scored.mod_avg) & np.isfinite(scored.pred)
        if mask.sum() > 1:
            y, p = scored.loc[mask, 'mod_avg'], scored.loc[mask, 'pred']
            trial_metrics[key] = {'n': int(mask.sum()), 'rho': float(spearmanr(y, p).statistic),
                                  'mse': float(mean_squared_error(y, p))}
    else:
        pred.columns = ['ContextID', 'pred_mod', 'pred_head']
        pred[['pred_mod', 'pred_head']] = pred[['pred_mod', 'pred_head']].apply(pd.to_numeric, errors='coerce')
        scored = gold.merge(pred, on='ContextID', how='inner')
        metric = {'n': int(len(scored))}
        for role in ('mod', 'head'):
            mask = np.isfinite(scored[f'{role}_avg']) & np.isfinite(scored[f'pred_{role}'])
            if mask.sum() > 1:
                y, p = scored.loc[mask, f'{role}_avg'], scored.loc[mask, f'pred_{role}']
                metric[f'{role}_rho'] = float(spearmanr(y, p).statistic)
                metric[f'{role}_mse'] = float(mean_squared_error(y, p))
        if len(metric) > 1:
            trial_metrics[key] = metric

metrics_path = work / 'submission' / 'trial_metrics.json'
metrics_path.write_text(json.dumps(trial_metrics, indent=2), encoding='utf-8')
print(json.dumps(trial_metrics, indent=2))
print('saved ->', metrics_path)

### LÆ°u Ã½
- TÃ­n hiá»‡u tá»‘t: `val_rho_mean` (rho trung bÃ¬nh Mod/Head trÃªn holdout 20%, split theo compound).
- Dedicated exits are always active: `gauss_ctx_mod=19`, `gauss_ctx_head=20`, and `gauss_ctx_pv=21,22`.
- Data mix: `TRAIN_MIX` = full (9825 rows, de-pv ~97% representation-only) / default (8335) / nn (6778).
- predict / train5 / resume chÆ°a wire trong `src/` (cháº¡y qua `mm/` cÅ© náº¿u cáº§n). CELL TRIAL chá»‰ tÃ¡i score holdout Ä‘á»ƒ xem prediction.